In [1]:
from decimal import Decimal, getcontext
import math
from functools import lru_cache

getcontext().prec = 100

### Задача 1.
Рассчитать с машинной точностью интеграл с неопределенностью
$$ \int_{-1}^{1} dx \frac{\sin(x/2)}{e^x - 1} $$
квадратурой на разностных схемах.

In [2]:
# из учебных материалов
@lru_cache(None)
def factorial_dec(n: int) -> Decimal:
    if n < 0:
        raise ValueError("Negative factorial not defined")
    if n == 0 or n == 1:
        return Decimal(1)
    return Decimal(n) * factorial_dec(n - 1)

@lru_cache(None)
def comb_dec(n: int, k: int) -> Decimal:
    if k < 0 or k > n:
        return Decimal(0)
    return factorial_dec(n) / (factorial_dec(k) * factorial_dec(n - k))

@lru_cache(None)
def D(n: int, j: int) -> Decimal:
    """
    Вычисляет D_n^(j) по заданной рекуррентной формуле (см. теорию).
    """
    if j == 0:
        return Decimal(1)
    # "прогреваем" кэш (рекурсия):
    _ = D(n, j-1)
    return _compute_all_D_up_to(n, j)[j]

@lru_cache(None)
def _compute_all_D_up_to(n: int, J: int):
    results = [Decimal(0)]*(J+1)
    results[0] = Decimal(1)
    for j in range(J):
        s = Decimal(0)
        for k in range(n + 2*j + 1):
            for l in range(j+1):
                val = (
                    Decimal((-1)**k)
                    * results[l]
                    * comb_dec(n+2*l, k - (j - l))
                    * Decimal((n + 2*j - 2*k)**(n + 2*j + 2))
                    / (Decimal(2)**(n + 2*j + 2) * factorial_dec(n + 2*j + 2))
                )
                s += val
        # Финальная формула берёт только s, без + results[j] (как в коде лекции)
        results[j+1] = s
    return results

@lru_cache(None)
def A(k: int, n: int, m: int) -> Decimal:
    """
    A_{k,n}^m = sum_{l=0}^m [ (-1)^(k-m) * D_n^(l) * C_{n+2l}^{k - m + l} ].
    """
    s = Decimal(0)
    sign = Decimal((-1)**(k - m))
    for l in range(m+1):
        s += (
            sign
            * D(n, l)
            * comb_dec(n+2*l, k - m + l)
        )
    return s

@lru_cache(None)
def W_list(m: int):
    """
    Коэффициенты W_k^m (k=0..2m) в Decimal.
    W_k^m = sum_{n=0}^m [ A_{k,2n}^{m-n} / (2^(2n)*(2n+1)!) ].
    """
    result = []
    for k in range(2*m + 1):
        s = Decimal(0)
        for n_ in range(m+1):
            a_val = A(k, 2*n_, m - n_)
            denom = (Decimal(2)**(2*n_) * factorial_dec(2*n_+1))
            s += a_val / denom
        result.append(s)
    return result

def build_function_values_diff_scheme(f, a: float, b: float, J: int, m: int):
    """
    Центральная сетка: x_j = a + (j+0.5)*h, j=-m..(J-1+m).
    Возвращает (y_values, h, calls).
    """
    h = (b - a)/J
    values = []
    # Диапазон индексов согласно теории, чтобы покрыть "вылезание" за края
    for j_real in range(-m, (J - 1) + m + 1):
        x_val = a + (j_real + 0.5)*h
        values.append(f(x_val))
    calls = len(values)
    return values, h, calls

def integrate_by_diff_scheme(f, a: float, b: float, J: int, m: int):
    """
    Разностная (дифференсная) квадратура порядка 2m:
    ∫[a..b] f(x) dx ≈ h Σ_j Σ_k [ W_{m-k} * f( x_{j+k} ) ].
    Возвращает (approx, calls).
    """
    y_values, h, calls = build_function_values_diff_scheme(f, a, b, J, m)
    W_dec = W_list(m)
    W = [float(wd) for wd in W_dec]

    offset = m
    total_sum = 0.0

    # Суммирование сверткой
    for j in range(J):
        loc_sum = 0.0
        for k in range(-m, m+1):
            idx = (j + k) + offset
            # W[m-k] соответствует индексации коэффициентов
            loc_sum += W[m - k] * y_values[idx]
        total_sum += loc_sum

    return (h * total_sum, calls)

In [5]:
def target_func(x: float) -> float:
    denom = math.expm1(x)
    if abs(denom) < 1e-19:
        return 0.5
    return math.sin(x/2) / denom


a, b = -1.0, 1.0
ref_val, _ = integrate_by_diff_scheme(target_func, a, b, J=64, m=11)
for m in [3, 5, 7]:
    print(f"Порядок схемы m={m}")
    for J in [4, 8, 16, 32]:
        approx, calls = integrate_by_diff_scheme(target_func, a, b, J, m)
        err = abs(approx - ref_val)
        print(f" J={J:2d}, calls={calls:3d} | Result={approx:.16f}, Error={err:.2e}")
    print()

Порядок схемы m=3
 J= 4, calls= 10 | Result=1.0130392166350388, Error=1.96e-08
 J= 8, calls= 14 | Result=1.0130392361530993, Error=7.95e-11
 J=16, calls= 22 | Result=1.0130392362323124, Error=3.14e-13
 J=32, calls= 38 | Result=1.0130392362326248, Error=1.33e-15

Порядок схемы m=5
 J= 4, calls= 14 | Result=1.0130392360748255, Error=1.58e-10
 J= 8, calls= 18 | Result=1.0130392362325813, Error=4.49e-14
 J=16, calls= 26 | Result=1.0130392362326259, Error=2.22e-16
 J=32, calls= 42 | Result=1.0130392362326262, Error=0.00e+00

Порядок схемы m=7
 J= 4, calls= 18 | Result=1.0130392362286704, Error=3.96e-12
 J= 8, calls= 22 | Result=1.0130392362326262, Error=0.00e+00
 J=16, calls= 30 | Result=1.0130392362326259, Error=2.22e-16
 J=32, calls= 46 | Result=1.0130392362326262, Error=0.00e+00



### Задача 2.
Используя методику прямой сшивки при взятии сумм рядов, рассчитать постоянную $\gamma_E$ Эйлера-Маскерони с ошибкой в 13 знаке по формуле
$$ \gamma_E = \lim_{n\to\infty} \left( 1 + 1/2 + 1/3 + 1/4 + \ldots + 1/n - \ln n \right) $$

In [6]:
getcontext().prec = 60

# из учебных материалов
@lru_cache(None)
def factorial_dec(n):
    if n <= 1: return Decimal(1)
    return Decimal(n) * factorial_dec(n - 1)

@lru_cache(None)
def comb_dec(n, k):
    if k < 0 or k > n: return Decimal(0)
    return factorial_dec(n) / (factorial_dec(k) * factorial_dec(n - k))

@lru_cache(None)
def D(n, j):
    if j == 0: return Decimal(1)
    # Внутренняя логика рекурсии D (упрощенно для краткости, полная версия была выше)
    # Для скорости просто вызовем precomputed если бы могли, но тут честно считаем:
    _ = D(n, j-1)
    # В этой функции главное вернуть корректное значение.
    # Используем полную реализацию из предыдущего шага:
    results = [Decimal(0)]*(j+1)
    results[0] = Decimal(1)
    for jj in range(j):
        s = Decimal(0)
        for k in range(n + 2*jj + 1):
            for l in range(jj+1):
                s += (Decimal((-1)**k) * results[l] * comb_dec(n+2*l, k-(jj-l)) *
                      Decimal((n+2*jj-2*k)**(n+2*jj+2)) /
                      (Decimal(2)**(n+2*jj+2) * factorial_dec(n+2*jj+2)))
        results[jj+1] = s
    return results[j]

@lru_cache(None)
def A(k, n, m):
    s = Decimal(0)
    sign = Decimal((-1)**(k - m))
    for l in range(m+1):
        s += sign * D(n, l) * comb_dec(n+2*l, k - m + l)
    return s

@lru_cache(None)
def W_list(m):
    result = []
    for k in range(2*m + 1):
        s = Decimal(0)
        for n_ in range(m+1):
            s += A(k, 2*n_, m - n_) / (Decimal(2)**(2*n_) * factorial_dec(2*n_+1))
        result.append(s)
    return result


def get_I_list(m):
    """
    I_l^m = sum_{k=0}^l W_{2m-k}
    """
    W = W_list(m)
    I = []
    for l in range(2*m + 1):
        val = Decimal(0)
        # Сумма W с конца (W_{2m}, W_{2m-1}...)
        for k in range(l + 1):
            val += W[2*m - k]
        I.append(val)
    return I

In [7]:
def series_term(n):
    """ a_n = 1/n - ln(1 + 1/n) """
    nd = Decimal(n)
    return Decimal(1)/nd - (Decimal(1) + Decimal(1)/nd).ln()

def tail_integral(x):
    """
    Интеграл от x до бесконечности для a(t) = 1/t - ln(1+1/t).
    Первообразная P(t) = 1 - (t+1)ln(1 + 1/t).
    int_x^inf = P(inf) - P(x) = 0 - (1 - (x+1)ln(1+1/x)) = (x+1)ln(1+1/x) - 1.
    """
    xd = Decimal(x)
    return (xd + 1) * (Decimal(1) + Decimal(1)/xd).ln() - Decimal(1)

def calc_gamma_direct_stitch(m=7, N_split=1000):
    head_sum = Decimal(0)
    for j in range(1, N_split - m + 1):
        head_sum += series_term(j)

    correction_sum = Decimal(0)
    I_coeffs = get_I_list(m)

    for k in range(2*m):
        j = N_split - m + 1 + k

        weight = Decimal(1) - I_coeffs[k]
        correction_sum += weight * series_term(j)
    integral_val = tail_integral(N_split + 0.5)

    return head_sum + correction_sum + integral_val

gamma_calc = calc_gamma_direct_stitch(m=7, N_split=1000)

print(f"Calculated Gamma: {gamma_calc:.15f}")
print(f"Known Value:      0.577215664901532")
print(f"Error:            {abs(gamma_calc - Decimal('0.577215664901532')):.2e}")

Calculated Gamma: 0.577215664901533
Known Value:      0.577215664901532
Error:            8.61e-16


### Задача 3.
Вычислить сумму ряда
$$ \sum_{n=1}^{\infty} \frac{n^2 + 1}{n^4 + n^2 + 1} \cos(2n) $$
с точностью $\varepsilon = 10^{-6}$, применяя метод Куммера.

In [ ]:
x = 2.0
S1 = (math.pi**2) / 6.0 - (math.pi * x) / 2.0 + (x**2) / 4.0

# для точности 10^-6 достаточно k >= 10.
N_iter = 20
correction_sum = 0.0

for n in range(1, N_iter + 1):
    # a_n (исходный) = (n^2+1)/(n^4+n^2+1) * cos(2n)
    # b_n (вспомогательный) = 1/n^2 * cos(2n)

    term = -math.cos(2*n) / (n**2 * (n**4 + n**2 + 1))
    correction_sum += term

total_sum = S1 + correction_sum

print(f"Вспомогательная сумма S1 (аналитически): {S1:.8f}")
print(f"Поправка (сумма разностного ряда):       {correction_sum:.8e}")
print(f"Итоговая сумма ряда:                     {total_sum:.8f}")

### Задача 4.
Взять предел
$$ \lim_{x\to 0} \frac{y(x) - 1}{x} $$
где
$$ y(x) = \begin{cases} \cosh\sqrt{|x|}, & x > 0 \\ \cos\sqrt{|x|}, & x < 0 \end{cases} $$
Поиграться с выбором шага.

In [10]:
def y(x):
    if x > 0:
        return math.cosh(math.sqrt(x))
    elif x < 0:
        return math.cos(math.sqrt(abs(x)))
    else:
        return 1.0

def g(x):
    if x == 0: return None
    return (y(x) - 1) / x

def f_sym(x):
    return (g(x) + g(-x)) / 2


print(f"{'Шаг (dx)':>10} | {'Значение f_sym(dx)':>20} | {'Ошибка':>10}")
print("-" * 50)

for power in range(1, 9):
    dx = 10**(-power)
    res = f_sym(dx)
    error = abs(res - 0.5)
    print(f"{dx:10.8f} | {res:20.15f} | {error:.2e}")

  Шаг (dx) |   Значение f_sym(dx) |     Ошибка
--------------------------------------------------
0.10000000 |    0.500013888916446 | 1.39e-05
0.01000000 |    0.500000138888884 | 1.39e-07
0.00100000 |    0.500000001388889 | 1.39e-09
0.00010000 |    0.500000000014378 | 1.44e-11
0.00001000 |    0.500000000003276 | 3.28e-12
0.00000100 |    0.499999999958867 | 4.11e-11
0.00000010 |    0.499999999736822 | 2.63e-10
0.00000001 |    0.499999996961265 | 3.04e-09
